In [35]:
import pandas as pd

df = pd.read_parquet("final_data/2025-01_Friday.parquet")
#df.describe()
df.head(100)

,month_period,day_of_week,route_name,direction,stop_id,stop_name,stop_sequence,hour,avg_delay_seconds,count
0,2025-01,Friday,1,Downtown/South,103S,238 St,2.0,0.0,-2.533333,15
1,2025-01,Friday,1,Downtown/South,103S,238 St,2.0,1.0,7.933333,15
2,2025-01,Friday,1,Downtown/South,103S,238 St,2.0,2.0,16.266667,15
3,2025-01,Friday,1,Downtown/South,103S,238 St,2.0,3.0,23.266667,15
4,2025-01,Friday,1,Downtown/South,103S,238 St,2.0,4.0,2.250000,20
...,...,...,...,...,...,...,...,...,...,...
95,2025-01,Friday,1,Downtown/South,120S,96 St,18.0,0.0,89.047619,21
96,2025-01,Friday,1,Downtown/South,120S,96 St,18.0,1.0,189.466667,15
97,2025-01,Friday,1,Downtown/South,120S,96 St,18.0,2.0,318.666667,15
98,2025-01,Friday,1,Downtown/South,120S,96 St,18.0,3.0,207.200000,15


In [4]:
import argparse
import os
import pandas as pd
from datetime import datetime, timedelta, date
from typing import Optional
from zoneinfo import ZoneInfo

In [ ]:
import os
import pandas as pd
from datetime import datetime, timedelta, date
from zoneinfo import ZoneInfo

NY = ZoneInfo("America/New_York")
UTC = ZoneInfo("UTC")

ACTUAL_PARQUET = "parquet_daily/2025-01-01_part2.parquet"   # one day stop-level
GTFS_DIR = "gtfs"                                # folder containing GTFS txt files
TOLERANCE_MIN = 12                                    # nearest-match window

In [ ]:
def route_from_trip_uid(trip_uid: str) -> str:
    # examples: 1735707600_7..S  -> "7"
    #           1735707600_GS.N01R -> "GS"
    try:
        after = str(trip_uid).split("_", 1)[1]
        return after.split(".", 1)[0]
    except Exception:
        return ""

def unix_to_ny(x):
    if pd.isna(x):
        return pd.NaT
    return datetime.fromtimestamp(int(x), tz=UTC).astimezone(NY)

def gtfs_time_to_dt(service_day: date, t: str):
    """Convert GTFS HH:MM:SS (can exceed 24:00:00) to NY datetime on service_day."""
    if pd.isna(t) or t == "":
        return None
    hh, mm, ss = t.split(":")
    h, m, s = int(hh), int(mm), int(ss)
    base = datetime(service_day.year, service_day.month, service_day.day, tzinfo=NY)
    return base + timedelta(hours=h, minutes=m, seconds=s)

def service_ids_for_date(calendar_df: pd.DataFrame, calendar_dates_df: pd.DataFrame | None, d: date) -> set[str]:
    ymd = int(d.strftime("%Y%m%d"))
    weekday_col = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"][d.weekday()]

    cal = calendar_df.copy()
    cal["start_date"] = cal["start_date"].astype(int)
    cal["end_date"] = cal["end_date"].astype(int)

    active = cal[
        (cal["start_date"] <= ymd) &
        (cal["end_date"] >= ymd) &
        (cal[weekday_col].astype(int) == 1)
    ]["service_id"]
    active_set = set(active.tolist())

    if calendar_dates_df is not None and len(calendar_dates_df) > 0:
        cd = calendar_dates_df.copy()
        cd["date"] = cd["date"].astype(int)
        todays = cd[cd["date"] == ymd][["service_id","exception_type"]]
        added = set(todays[todays["exception_type"].astype(int) == 1]["service_id"].tolist())
        removed = set(todays[todays["exception_type"].astype(int) == 2]["service_id"].tolist())
        active_set |= added
        active_set -= removed

    return active_set


In [ ]:
actual = pd.read_parquet(ACTUAL_PARQUET)[["trip_uid","stop_id","arrival_time","departure_time"]].copy()
actual["route_id"] = actual["trip_uid"].apply(route_from_trip_uid)

actual["actual_arr_dt"] = actual["arrival_time"].apply(unix_to_ny)
actual["actual_dep_dt"] = actual["departure_time"].apply(unix_to_ny)

# build events (arrival + departure)
arr = actual[actual["actual_arr_dt"].notna()].copy()
arr["event_kind"] = "arrival"
arr["actual_dt"] = arr["actual_arr_dt"]

dep = actual[actual["actual_dep_dt"].notna()].copy()
dep["event_kind"] = "departure"
dep["actual_dt"] = dep["actual_dep_dt"]

actual_events = pd.concat([arr, dep], ignore_index=True)[
    ["trip_uid","route_id","stop_id","event_kind","actual_dt"]
]

# service date candidates (handles after-midnight)
actual_events["service_date_0"] = actual_events["actual_dt"].dt.date
actual_events["service_date_1"] = actual_events["service_date_0"].apply(lambda d: d - timedelta(days=1))

dates_needed = set(actual_events["service_date_0"]) | set(actual_events["service_date_1"])
print("Dates needed:", sorted(dates_needed)[:5], "...")
print("Actual events:", len(actual_events))
actual_events.head()


Dates needed: [datetime.date(2024, 12, 30), datetime.date(2024, 12, 31), datetime.date(2025, 1, 1), datetime.date(2025, 1, 2)] ...
Actual events: 312021


,trip_uid,route_id,stop_id,event_kind,actual_dt,service_date_0,service_date_1
0,1735707600_7..S,7,702S,arrival,2025-01-01 05:02:10-05:00,2025-01-01,2024-12-31
1,1735707600_7..S,7,705S,arrival,2025-01-01 05:03:57-05:00,2025-01-01,2024-12-31
2,1735707600_7..S,7,706S,arrival,2025-01-01 05:05:16-05:00,2025-01-01,2024-12-31
3,1735707600_7..S,7,707S,arrival,2025-01-01 05:08:53-05:00,2025-01-01,2024-12-31
4,1735707600_7..S,7,708S,arrival,2025-01-01 05:10:30-05:00,2025-01-01,2024-12-31


In [ ]:
import os

# Build schedule_events from GTFS for all dates we need
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
calendar_df = pd.read_csv(os.path.join(GTFS_DIR, "calendar.txt"))

calendar_dates_path = os.path.join(GTFS_DIR, "calendar_dates.txt")
calendar_dates_df = pd.read_csv(calendar_dates_path) if os.path.exists(calendar_dates_path) else None

# Ensure string types where needed
trips["service_id"] = trips["service_id"].astype(str)
stop_times["trip_id"] = stop_times["trip_id"].astype(str)
calendar_df["service_id"] = calendar_df["service_id"].astype(str)
if calendar_dates_df is not None:
    calendar_dates_df["service_id"] = calendar_dates_df["service_id"].astype(str)

# dates_needed comes from the earlier actual_events cell
all_days = sorted(dates_needed)

schedule_rows: list[pd.DataFrame] = []

for d in all_days:
    active_services = service_ids_for_date(calendar_df, calendar_dates_df, d)
    if not active_services:
        continue

    trips_d = trips[trips["service_id"].isin(active_services)].copy()
    if trips_d.empty:
        continue

    # Join trips to stop_times to get route_id with each stop
    st = stop_times.merge(
        trips_d[["trip_id", "route_id", "service_id"]],
        on="trip_id",
        how="inner",
    )

    # Arrival events
    arr = st[st["arrival_time"].notna() & (st["arrival_time"] != "")].copy()
    arr["event_kind"] = "arrival"
    arr["scheduled_dt"] = arr["arrival_time"].apply(lambda t: gtfs_time_to_dt(d, t))

    # Departure events
    dep = st[st["departure_time"].notna() & (st["departure_time"] != "")].copy()
    dep["event_kind"] = "departure"
    dep["scheduled_dt"] = dep["departure_time"].apply(lambda t: gtfs_time_to_dt(d, t))

    day_ev = pd.concat([arr, dep], ignore_index=True)
    day_ev["service_date"] = d

    # Columns expected by the matching code
    day_ev = day_ev[[
        "service_date",
        "route_id",
        "trip_id",
        "stop_id",
        "stop_sequence",
        "event_kind",
        "scheduled_dt",
    ]]

    # Drop rows where conversion failed
    day_ev = day_ev[day_ev["scheduled_dt"].notna()]

    schedule_rows.append(day_ev)

if schedule_rows:
    schedule_events = pd.concat(schedule_rows, ignore_index=True)
else:
    schedule_events = pd.DataFrame(
        columns=[
            "service_date",
            "route_id",
            "trip_id",
            "stop_id",
            "stop_sequence",
            "event_kind",
            "scheduled_dt",
        ]
    )

print("Schedule events rows:", len(schedule_events))
schedule_events.head()

Schedule events rows: 1750666


,service_date,route_id,trip_id,stop_id,stop_sequence,event_kind,scheduled_dt
0,2024-12-30,1,AFA24GEN-1093-Weekday-00_000650_1..S03R,101S,1,arrival,2024-12-30 00:06:30-05:00
1,2024-12-30,1,AFA24GEN-1093-Weekday-00_000650_1..S03R,103S,2,arrival,2024-12-30 00:08:00-05:00
2,2024-12-30,1,AFA24GEN-1093-Weekday-00_000650_1..S03R,104S,3,arrival,2024-12-30 00:09:30-05:00
3,2024-12-30,1,AFA24GEN-1093-Weekday-00_000650_1..S03R,106S,4,arrival,2024-12-30 00:11:00-05:00
4,2024-12-30,1,AFA24GEN-1093-Weekday-00_000650_1..S03R,107S,5,arrival,2024-12-30 00:12:30-05:00


In [ ]:
tol = pd.Timedelta(minutes=TOLERANCE_MIN)

# Ensure proper dtypes
actual_events["actual_dt"] = pd.to_datetime(actual_events["actual_dt"])
schedule_events["scheduled_dt"] = pd.to_datetime(schedule_events["scheduled_dt"])

actual_events["service_date_0"] = pd.to_datetime(actual_events["service_date_0"]).dt.date
actual_events["service_date_1"] = pd.to_datetime(actual_events["service_date_1"]).dt.date
schedule_events["service_date"] = pd.to_datetime(schedule_events["service_date"]).dt.date

KEYS = ["route_id", "stop_id", "event_kind", "service_date"]

def groupwise_asof(left_df: pd.DataFrame, right_df: pd.DataFrame) -> pd.DataFrame:
    """
    left_df must have columns: route_id, stop_id, event_kind, service_date, actual_dt
    right_df must have columns: route_id, stop_id, event_kind, service_date, scheduled_dt
    Returns left rows with nearest scheduled match in the same group.
    """
    # Pre-split right into dict for fast lookup
    right_groups = {}
    for k, g in right_df.groupby(KEYS, sort=False):
        right_groups[k] = g.sort_values("scheduled_dt").reset_index(drop=True)

    out = []
    for k, g in left_df.groupby(KEYS, sort=False):
        g = g.sort_values("actual_dt").reset_index(drop=True)
        rg = right_groups.get(k)
        if rg is None or rg.empty:
            # no schedule for this group
            gg = g.copy()
            gg["scheduled_dt"] = pd.NaT
            gg["trip_id"] = pd.NA
            gg["stop_sequence"] = pd.NA
            out.append(gg)
            continue

        m = pd.merge_asof(
            g,
            rg,
            left_on="actual_dt",
            right_on="scheduled_dt",
            tolerance=tol,
            direction="nearest",
            allow_exact_matches=True,
            suffixes=("", "_sched"),
        )
        out.append(m)

    return pd.concat(out, ignore_index=True)

# --- Attempt 1: service_date_0 ---
left0 = actual_events.rename(columns={"service_date_0": "service_date"}).copy()
left0 = left0[["trip_uid","route_id","stop_id","event_kind","actual_dt","service_date"]]

right = schedule_events[["service_date","route_id","trip_id","stop_id","stop_sequence","event_kind","scheduled_dt"]].copy()

m0 = groupwise_asof(left0, right)
m0["matched_using"] = "service_date_0"

# --- Attempt 2: fill unmatched using service_date_1 ---
need = m0["scheduled_dt"].isna()
if need.any():
    left1 = actual_events.rename(columns={"service_date_1": "service_date"}).copy()
    left1 = left1[["trip_uid","route_id","stop_id","event_kind","actual_dt","service_date"]]

    m1 = groupwise_asof(left1, right)
    m1["matched_using"] = "service_date_1"

    # key to align rows for replacement
    keycols = ["trip_uid","route_id","stop_id","event_kind","actual_dt"]
    m0["k"] = m0[keycols].astype(str).agg("|".join, axis=1)
    m1["k"] = m1[keycols].astype(str).agg("|".join, axis=1)
    m1_map = m1.set_index("k")

    replaced = []
    for _, row in m0.iterrows():
        if pd.notna(row["scheduled_dt"]):
            replaced.append(row)
            continue
        k = row["k"]
        if k in m1_map.index and pd.notna(m1_map.loc[k, "scheduled_dt"]):
            replaced.append(m1_map.loc[k])
        else:
            replaced.append(row)

    joined = pd.DataFrame(replaced).drop(columns=["k"])
else:
    joined = m0

joined["delay_seconds"] = (joined["actual_dt"] - joined["scheduled_dt"]).dt.total_seconds()

print("Matched rate:", joined["scheduled_dt"].notna().mean())
joined.head()


/var/folders/3z/j_4_m2s54119gs_5zw24ryf40000gn/T/ipykernel_52345/2896082170.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(out, ignore_index=True)
/var/folders/3z/j_4_m2s54119gs_5zw24ryf40000gn/T/ipykernel_52345/2896082170.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(out, ignore_index=True)


Matched rate: 0.9881257992250522


,trip_uid,route_id,stop_id,event_kind,actual_dt,service_date,service_date_sched,route_id_sched,trip_id,stop_id_sched,stop_sequence,event_kind_sched,scheduled_dt,matched_using,delay_seconds
1735775880_7..S|7|702S|arrival|2025-01-01 00:00:10-05:00,1735775880_7..S,7,702S,arrival,2025-01-01 00:00:10-05:00,2024-12-31,2024-12-31,7,L0S1-7-1064-S02_143900_7..S97R,702S,2.0,arrival,2025-01-01 00:01:30-05:00,service_date_1,-80.0
1,1735707600_7..S,7,702S,arrival,2025-01-01 05:02:10-05:00,2025-01-01,2025-01-01,7,L0S3-7-3026-S02_030000_7..S97R,702S,2.0,arrival,2025-01-01 05:02:30-05:00,service_date_0,-20.0
2,1735708800_7..S,7,702S,arrival,2025-01-01 05:22:10-05:00,2025-01-01,2025-01-01,7,L0S3-7-3026-S02_032000_7..S97R,702S,2.0,arrival,2025-01-01 05:22:30-05:00,service_date_0,-20.0
3,1735710000_7..S,7,702S,arrival,2025-01-01 05:42:10-05:00,2025-01-01,2025-01-01,7,L0S3-7-3026-S02_033700_7..S97R,702S,2.0,arrival,2025-01-01 05:39:30-05:00,service_date_0,160.0
4,1735711200_7..S,7,702S,arrival,2025-01-01 06:02:10-05:00,2025-01-01,2025-01-01,7,L0S3-7-3026-S02_036000_7..S97R,702S,2.0,arrival,2025-01-01 06:02:30-05:00,service_date_0,-20.0


In [ ]:
df = joined.copy()

df = df[[
    "trip_uid",
    "route_id",
    "stop_id",
    "stop_sequence",
    "actual_dt",
    "scheduled_dt",
    "delay_seconds"
]].copy()

df["hour"] = df["scheduled_dt"].dt.hour
df["day_of_week"] = df["scheduled_dt"].dt.dayofweek
df["is_weekend"] = df["day_of_week"] >= 5

df["scheduled_headway_min"] = (
    df.sort_values("scheduled_dt")
      .groupby(["route_id","stop_id"])["scheduled_dt"]
      .diff()
      .dt.total_seconds() / 60
)

df = df[df["delay_seconds"].abs() < 1800]   # remove >30 min
df["delay_seconds"].describe()


count    308316.000000
mean          8.809520
std         154.553773
min        -720.000000
25%         -63.000000
50%           0.000000
75%          85.000000
max         718.000000
Name: delay_seconds, dtype: float64

In [ ]:
import pandas as pd
import os

df = joined.copy()

keep_cols = [
    "trip_uid", "route_id", "stop_id", "stop_sequence",
    "event_kind", "actual_dt", "scheduled_dt", "delay_seconds"
]
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()

stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"), dtype=str)

stops_small = stops[["stop_id", "stop_name", "parent_station", "stop_lat", "stop_lon"]].copy()
df = df.merge(stops_small, on="stop_id", how="left")

df["direction"] = df["stop_id"].str[-1].map({"N": "Uptown/North", "S": "Downtown/South"})


route_map = {
    "GS": "Grand Central Shuttle",
    "FS": "Franklin Av Shuttle",
    "H":  "Rockaway Park Shuttle",
}
df["route_name"] = df["route_id"].replace(route_map)

df["service_date"] = df["scheduled_dt"].dt.date
df["scheduled_time"] = df["scheduled_dt"].dt.strftime("%H:%M:%S")
df["actual_time"] = df["actual_dt"].dt.strftime("%H:%M:%S")
df["hour"] = df["scheduled_dt"].dt.hour
df["day_of_week"] = df["scheduled_dt"].dt.day_name()

parent_lookup = stops[stops["location_type"].fillna("").astype(str).eq("1")][["stop_id", "stop_name"]].copy()
parent_lookup = parent_lookup.rename(columns={"stop_id": "parent_station", "stop_name": "parent_stop_name"})
df = df.merge(parent_lookup, on="parent_station", how="left")

readable = df[[
    "service_date",
    "route_name",
    "direction",
    "stop_id",
    "stop_name",
    "parent_stop_name",
    "stop_sequence",
    "event_kind",
    "scheduled_dt",
    "actual_dt",
    "delay_seconds",
    "scheduled_time",
    "actual_time",
    "hour",
    "day_of_week",
    "trip_uid",
]]

readable.head(25)


,service_date,route_name,direction,stop_id,stop_name,parent_stop_name,stop_sequence,event_kind,scheduled_dt,actual_dt,delay_seconds,scheduled_time,actual_time,hour,day_of_week,trip_uid
0,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 00:01:30-05:00,2025-01-01 00:00:10-05:00,-80.0,00:01:30,00:00:10,0.0,Wednesday,1735775880_7..S
1,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 05:02:30-05:00,2025-01-01 05:02:10-05:00,-20.0,05:02:30,05:02:10,5.0,Wednesday,1735707600_7..S
2,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 05:22:30-05:00,2025-01-01 05:22:10-05:00,-20.0,05:22:30,05:22:10,5.0,Wednesday,1735708800_7..S
3,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 05:39:30-05:00,2025-01-01 05:42:10-05:00,160.0,05:39:30,05:42:10,5.0,Wednesday,1735710000_7..S
4,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:02:30-05:00,2025-01-01 06:02:10-05:00,-20.0,06:02:30,06:02:10,6.0,Wednesday,1735711200_7..S
5,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:11:30-05:00,2025-01-01 06:11:27-05:00,-3.0,06:11:30,06:11:27,6.0,Wednesday,1735711740_7..S
6,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:19:30-05:00,2025-01-01 06:19:10-05:00,-20.0,06:19:30,06:19:10,6.0,Wednesday,1735712220_7..S
7,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:27:30-05:00,2025-01-01 06:27:10-05:00,-20.0,06:27:30,06:27:10,6.0,Wednesday,1735712700_7..S
8,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:35:30-05:00,2025-01-01 06:35:10-05:00,-20.0,06:35:30,06:35:10,6.0,Wednesday,1735713180_7..S
9,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:43:30-05:00,2025-01-01 06:43:10-05:00,-20.0,06:43:30,06:43:10,6.0,Wednesday,1735713660_7..S


In [ ]:
OUT_PATH = "joined_readable_2025-01-01.parquet"
readable.to_parquet(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Saved: joined_readable_2025-01-01.parquet


In [ ]:
df = pd.read_parquet("joined_readable_2025-01-01.parquet")
df.head(25)

,service_date,route_name,direction,stop_id,stop_name,parent_stop_name,stop_sequence,event_kind,scheduled_dt,actual_dt,delay_seconds,scheduled_time,actual_time,hour,day_of_week,trip_uid
0,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 00:01:30-05:00,2025-01-01 00:00:10-05:00,-80.0,00:01:30,00:00:10,0.0,Wednesday,1735775880_7..S
1,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 05:02:30-05:00,2025-01-01 05:02:10-05:00,-20.0,05:02:30,05:02:10,5.0,Wednesday,1735707600_7..S
2,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 05:22:30-05:00,2025-01-01 05:22:10-05:00,-20.0,05:22:30,05:22:10,5.0,Wednesday,1735708800_7..S
3,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 05:39:30-05:00,2025-01-01 05:42:10-05:00,160.0,05:39:30,05:42:10,5.0,Wednesday,1735710000_7..S
4,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:02:30-05:00,2025-01-01 06:02:10-05:00,-20.0,06:02:30,06:02:10,6.0,Wednesday,1735711200_7..S
5,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:11:30-05:00,2025-01-01 06:11:27-05:00,-3.0,06:11:30,06:11:27,6.0,Wednesday,1735711740_7..S
6,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:19:30-05:00,2025-01-01 06:19:10-05:00,-20.0,06:19:30,06:19:10,6.0,Wednesday,1735712220_7..S
7,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:27:30-05:00,2025-01-01 06:27:10-05:00,-20.0,06:27:30,06:27:10,6.0,Wednesday,1735712700_7..S
8,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:35:30-05:00,2025-01-01 06:35:10-05:00,-20.0,06:35:30,06:35:10,6.0,Wednesday,1735713180_7..S
9,2025-01-01,7,Downtown/South,702S,Mets-Willets Point,Mets-Willets Point,2.0,arrival,2025-01-01 06:43:30-05:00,2025-01-01 06:43:10-05:00,-20.0,06:43:30,06:43:10,6.0,Wednesday,1735713660_7..S
